<a href="https://colab.research.google.com/github/adityaprasad2005/ME646-Turbulence_MLP_project/blob/main/mlp_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# We train a small MLP model on the dataset.txt

In [3]:
# import libs
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import  torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
import torch.nn.functional as F


In [5]:
# Mount the Google drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
file_path = r'/content/drive/MyDrive/Turbulence_project_dataset/dataset.txt'

data = pd.read_csv(file_path, sep='\t')

data.head()

,Unnamed: 0,major_axis,minor_axis,Aspect_Ratio,init_velocity,Angle_of_attack,Reynolds_Number,freq_m,Strouhal_Number
0,0,1,1.0,1,0.0001,0,100,0.000020,0.199967
1,1,1,1.0,1,0.0005,0,500,0.000117,0.233294
2,2,1,1.0,1,0.0010,0,1000,0.000250,0.250000
3,3,1,0.5,2,0.0001,0,100,0.000000,0.000000
4,4,1,0.5,2,0.0005,0,500,0.000175,0.349912


In [21]:
# We train a 2 layer MLP with 4 neurons in each layer with ReLU activation function

# Define the model

class MLP_model(nn.Module):
    def __init__(self):
        super(MLP_model, self).__init__()
        self.fc1 = nn.Linear(3, 4)
        self.fc2 = nn.Linear(4, 1)
        self.relu = F.relu

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x


model = MLP_model()

model_save_path = 'model_best_weights'

# Define the loss function and optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Training, validation and testing dataset
X = data[['Aspect_Ratio', 'Angle_of_attack', 'Reynolds_Number' ]]
y = data['Strouhal_Number']


In [22]:
# If the model has been previously trained, load the best weights
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.load_state_dict(torch.load(model_save_path, map_location=device))

<All keys matched successfully>

In [15]:
X

,Aspect_Ratio,Angle_of_attack,Reynolds_Number
0,1,0,100
1,1,0,500
2,1,0,1000
3,2,0,100
4,2,0,500
5,2,0,1000
6,2,5,100
7,2,5,500
8,2,5,1000
9,2,10,100


In [16]:
# training and testing dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.05, random_state=42)

# Convert to tensor
X_train = torch.tensor(X_train.values).float()
y_train = torch.tensor(y_train.values).float()
X_test = torch.tensor(X_test.values).float()
y_test = torch.tensor(y_test.values).float()
X_val = torch.tensor(X_val.values).float()
y_val = torch.tensor(y_val.values).float()

# print the sizes of the train, validation, test sets
print("Train size:", X_train.shape[0])
print("Validation size:", X_val.shape[0])
print("Test size:", X_test.shape[0])

Train size: 15
Validation size: 1
Test size: 5


In [17]:
max_epochs = 100

for i in range(max_epochs):
    model.zero_grad()
    output = model(X_train)
    loss =  criterion(output, y_train)
    loss.backward()
    optimizer.step()

    print(f"Epoch {i+1}/{max_epochs}, Loss: {loss.item()}")

with torch.no_grad():
    output = model(X_test)
    test_loss = criterion(output, y_test)
    print(f"Test Loss: {test_loss.item()}")


Epoch 1/100, Loss: 1036.3944091796875
Epoch 2/100, Loss: 710.3981323242188
Epoch 3/100, Loss: 464.60589599609375
Epoch 4/100, Loss: 286.5947265625
Epoch 5/100, Loss: 163.59437561035156
Epoch 6/100, Loss: 90.69434356689453
Epoch 7/100, Loss: 50.2253303527832
Epoch 8/100, Loss: 24.62667465209961
Epoch 9/100, Loss: 9.7815580368042
Epoch 10/100, Loss: 3.233579635620117
Epoch 11/100, Loss: 1.9112601280212402
Epoch 12/100, Loss: 1.2763102054595947
Epoch 13/100, Loss: 0.8617218136787415
Epoch 14/100, Loss: 0.5905336737632751
Epoch 15/100, Loss: 0.4127400517463684
Epoch 16/100, Loss: 0.29581984877586365
Epoch 17/100, Loss: 0.21871398389339447
Epoch 18/100, Loss: 0.1681298315525055
Epoch 19/100, Loss: 0.13440853357315063
Epoch 20/100, Loss: 0.11193232983350754
Epoch 21/100, Loss: 0.10020320117473602
Epoch 22/100, Loss: 0.0978207141160965
Epoch 23/100, Loss: 0.09642127901315689
Epoch 24/100, Loss: 0.09513627737760544
Epoch 25/100, Loss: 0.09395383298397064
Epoch 26/100, Loss: 0.09286344051361084

/usr/local/lib/python3.11/dist-packages/torch/nn/modules/loss.py:610: UserWarning: Using a target size (torch.Size([15])) that is different to the input size (torch.Size([15, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/loss.py:610: UserWarning: Using a target size (torch.Size([5])) that is different to the input size (torch.Size([5, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


In [23]:
# Prediction from the Model
with torch.no_grad():
  output = model(X_val)
  loss = criterion(output, y_val)

print("Predicting from the trained model:\n")

X_val_np = X_val.squeeze().detach().numpy()
y_val_np = y_val.squeeze().detach().numpy()
aspect_ratio = X_val_np[0]
angle_of_attack = X_val_np[1]
Reynolds_Number = X_val_np[2]


print(f"Input parameters : Aspect Ratio: {aspect_ratio}, Angle of Attack: {angle_of_attack}, Reynolds Number: {Reynolds_Number}")
print(f"Actual Strouhal Number: {y_val.squeeze().detach().numpy()}")
print(f"Predicted Strouhal Number: {output.squeeze().detach().numpy()}")
# print(f"MSE Error: {loss.item()}")
print(f"Percentage Error: {100*loss.item()/y_val_np}%\n\n")

Predicting from the trained model:

Input parameters : Aspect Ratio: 2.0, Angle of Attack: 0.0, Reynolds Number: 1000.0
Actual Strouhal Number: 0.37137550115585327
Predicted Strouhal Number: 0.2781752049922943
Percentage Error: 2.33895206451416%




/usr/local/lib/python3.11/dist-packages/torch/nn/modules/loss.py:610: UserWarning: Using a target size (torch.Size([1])) that is different to the input size (torch.Size([1, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


In [20]:
# save the trained model
torch.save(model.state_dict(), model_save_path)
